In [0]:
%pip install databricks-vectorsearch databricks-sdk langchain mlflow mlflow[databricks] langchain-community

In [0]:
dbutils.library.restartPython()

In [0]:
import os


os.environ['WORKSPACE_URL'] = dbutils.secrets.get('rag_llm_scope', 'db_host')
os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get('rag_llm_scope', 'db_agent_app_token')

vector_search_endpoint = 'agent_db'
vector_search_index = 'llm.rag.idx_docs_text'






In [0]:
from databricks.vector_search.client import VectorSearchClient
from langchain_community.vectorstores import DatabricksVectorSearch
from langchain_community.embeddings import DatabricksEmbeddings

def get_retriver():

    embedding_model = DatabricksEmbeddings(endpoint= 'databricks-bge-large-en')

    client = VectorSearchClient(
        workspace_url=os.environ['WORKSPACE_URL'],
        personal_access_token=os.environ['DATABRICKS_TOKEN']
    )
    vs_index = client.get_index(endpoint_name=vector_search_endpoint, index_name = vector_search_index)

    vector_store = DatabricksVectorSearch(
        vs_index,
        embedding=embedding_model,
        # The column name in the index that contains the text data to be embedded
        text_column="text"
    )
    return vector_store.as_retriever()



In [0]:
from langchain.chains import RetrievalQA